In [2]:
# Librerías
import pandas as pd
from pathlib import Path
import ee

/home/ddayann/miniconda3/envs/tf_cuda_gee/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### Mecanismo de autorización
1. Al ejecutar las siguientes líneas se habilitará un link de acceso
2. Asegura los permisos apropiados para google Earth Engine y demás necesarios
3. Se genera un Token
4. Copiar y pegar el token en la ventana emergente que aparece en la ventana superior de VSC

* Para habilitar nuevos usuarios es necesario desmarcar "ee.Authenticate(force=True)" - Actualmente ya tiene habilitado el ingreso del usuario presente.  
(4/1AeoWuM9_9T1gCVAvTELP8S2YXcFW26H8_p7npV3xK4wc_OUw6Fb5i68ThS8)

In [3]:
# Log Google Earth Engine
# ee.Authenticate(force=True)
ee.Initialize(project='ee-cafe-494102')

In [4]:
# Cargar asset de municipios
municipios_ee = ee.FeatureCollection("projects/ee-cafe-494102/assets/municipios")

# Validación
print("Número de municipios:", municipios_ee.size().getInfo())
print("Columnas:", municipios_ee.first().propertyNames().getInfo())

Número de municipios: 27
Columnas: ['OBJECTID', 'MpNombre', 'Depto', 'MpCodigo', 'MpCategor', 'SHAPE_Leng', 'MpAltitud', 'SHAPE_Area', 'MpArea', 'system:index', 'MpNorma', 'Restriccio']


In [5]:
# Fechas de análisis
fecha_inicio = "2005-01-01"
fecha_fin = "2026-01-01"

In [6]:
# Cargar TerraClimate mensual

terraclimate_et = (
    ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate(fecha_inicio, fecha_fin)
    .select(["aet", "pet"])                         # aet = evapotranspiración real mensual
                                                    # pet = evapotranspiración potencial mensual
)

In [7]:
def reducir_evapotranspiracion_mensual(img):
    fecha = ee.Date(img.get("system:time_start"))
    fecha_str = fecha.format("YYYY-MM-dd")
    anio = fecha.get("year")
    mes = fecha.get("month")

    # TerraClimate entrega aet y pet en mm con factor de escala 0.1
    et = img.select("aet").multiply(0.1).rename("et_real_mm")
    pet = img.select("pet").multiply(0.1).rename("et_potencial_mm")

    img_et = et.addBands(pet)

    reduccion = img_et.reduceRegions(
        collection=municipios_ee,
        reducer=ee.Reducer.mean()
            .combine(
                reducer2=ee.Reducer.count(),
                sharedInputs=True
            ),
        scale=4638
    )

    def limpiar(feat):
        return ee.Feature(None, {
            "municipio": feat.get("MpNombre"),
            "date": fecha_str,
            "anio": anio,
            "mes": mes,
            "et_real_mm": feat.get("et_real_mm_mean"),
            "et_potencial_mm": feat.get("et_potencial_mm_mean"),
            "n_pixeles": feat.get("et_real_mm_count")
        })

    return reduccion.map(limpiar)

In [8]:
evapotranspiracion_municipal = (
    terraclimate_et
    .map(reducir_evapotranspiracion_mensual)
    .flatten()
)

In [9]:
# Exportar a Google Drive sin geometría
task = ee.batch.Export.table.toDrive(
    collection=evapotranspiracion_municipal,
    description="evapotranspiracion_municipal_terraclimate_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="evapotranspiracion_municipal_terraclimate_2005_2025",
    fileFormat="CSV",
    selectors=[
        "municipio",
        "date",
        "anio",
        "mes",
        "et_real_mm",
        "et_potencial_mm",
        "n_pixeles"
    ]
)

task.start()

print("Exportación iniciada.")
print("Revisa el estado de la tarea en Google Earth Engine Tasks.")

Exportación iniciada.
Revisa el estado de la tarea en Google Earth Engine Tasks.


#### Validación del proceso 
A fin de validar que el proceso se está ejecutando es necesario abrir el proyecto de Google Earth Engine, en la pestaña Task. Es posible ver el avance del proceso.  
Al final del procesamiento el archivo será guardado en Google Drive: "/MyDrive/Gee_exports"

In [17]:
# Lectura archivo CSV exportado
BASE_DIR = Path.cwd().parents[1]
ruta_et = BASE_DIR / "data" / "raw" / "evapotranspiracion_municipal_terraclimate_2005_2025.csv"
df_et = pd.read_csv(ruta_et)
df_et

,municipio,date,anio,mes,et_real_mm,et_potencial_mm,n_pixeles
0,Villamaria,2005-01-01,2005,1,71.374261,74.759849,33
1,Risaralda,2005-01-01,2005,1,100.449261,101.255915,15
2,Neira,2005-01-01,2005,1,90.045290,90.843116,33
3,Anserma,2005-01-01,2005,1,99.382713,100.703081,21
4,Manzanares,2005-01-01,2005,1,90.139206,90.139206,18
...,...,...,...,...,...,...,...
6475,Belalcázar,2024-12-01,2024,12,94.412094,94.412094,14
6476,San José,2024-12-01,2024,12,94.935270,94.935270,7
6477,Marquetalia,2024-12-01,2024,12,94.043123,94.043123,11
6478,Norcasia,2024-12-01,2024,12,106.718529,106.718529,24


In [18]:
df_et.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   municipio        6480 non-null   object 
 1   date             6480 non-null   object 
 2   anio             6480 non-null   int64  
 3   mes              6480 non-null   int64  
 4   et_real_mm       6480 non-null   float64
 5   et_potencial_mm  6480 non-null   float64
 6   n_pixeles        6480 non-null   int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 354.5+ KB


In [19]:
# Tipificación
df_et["municipio"] = df_et["municipio"].astype("category")
df_et["date"] = pd.to_datetime(df_et["date"])

In [20]:
df_et.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   municipio        6480 non-null   category      
 1   date             6480 non-null   datetime64[ns]
 2   anio             6480 non-null   int64         
 3   mes              6480 non-null   int64         
 4   et_real_mm       6480 non-null   float64       
 5   et_potencial_mm  6480 non-null   float64       
 6   n_pixeles        6480 non-null   int64         
dtypes: category(1), datetime64[ns](1), float64(2), int64(3)
memory usage: 311.5 KB


In [23]:
# Ajustar fecha al último día del mes
df_et["date"] = pd.to_datetime(
    df_et["anio"].astype(str) + "-" +
    df_et["mes"].astype(str) + "-01"
) + pd.offsets.MonthEnd(0)

# Ordenar columnas
et_mes = df_et[
    [
        "municipio",
        "date",
        "anio",
        "mes",
        "et_real_mm",
        "et_potencial_mm",
        "n_pixeles"
    ]
].sort_values(["municipio", "anio", "mes"])

In [24]:
et_mes

,municipio,date,anio,mes,et_real_mm,et_potencial_mm,n_pixeles
15,Aguadas,2005-01-31,2005,1,93.492632,95.107386,38
42,Aguadas,2005-02-28,2005,2,86.827158,87.810316,38
69,Aguadas,2005-03-31,2005,3,106.872298,106.942018,38
96,Aguadas,2005-04-30,2005,4,97.115719,97.115719,38
123,Aguadas,2005-05-31,2005,5,94.956035,94.956035,38
...,...,...,...,...,...,...,...
6362,Viterbo,2024-08-31,2024,8,112.075019,112.075019,11
6389,Viterbo,2024-09-30,2024,9,114.525354,114.525354,11
6416,Viterbo,2024-10-31,2024,10,103.556898,103.556898,11
6443,Viterbo,2024-11-30,2024,11,96.225354,96.225354,11


In [25]:
# Agregación anual
et_anual = (
    et_mes
    .groupby(["municipio", "anio"], observed=False)
    .agg(
        et_real_mm=("et_real_mm", "sum"),
        et_potencial_mm=("et_potencial_mm", "sum"),
        n_pixeles=("n_pixeles", "mean")
    )
    .reset_index()
)

et_anual["date"] = pd.to_datetime(
    et_anual["anio"].astype(str) + "-12-31"
)

et_anual = et_anual[
    [
        "municipio",
        "date",
        "anio",
        "et_real_mm",
        "et_potencial_mm",
        "n_pixeles"
    ]
]

In [27]:
display(et_mes.head(12))
display(et_anual.head(21))

,municipio,date,anio,mes,et_real_mm,et_potencial_mm,n_pixeles
15,Aguadas,2005-01-31,2005,1,93.492632,95.107386,38
42,Aguadas,2005-02-28,2005,2,86.827158,87.810316,38
69,Aguadas,2005-03-31,2005,3,106.872298,106.942018,38
96,Aguadas,2005-04-30,2005,4,97.115719,97.115719,38
123,Aguadas,2005-05-31,2005,5,94.956035,94.956035,38
150,Aguadas,2005-06-30,2005,6,83.753298,83.753298,38
177,Aguadas,2005-07-31,2005,7,97.569632,100.402105,38
204,Aguadas,2005-08-31,2005,8,97.310649,97.310649,38
231,Aguadas,2005-09-30,2005,9,96.870228,96.870228,38
258,Aguadas,2005-10-31,2005,10,85.035895,85.035895,38


,municipio,date,anio,et_real_mm,et_potencial_mm,n_pixeles
0,Aguadas,2005-12-31,2005,1107.821281,1113.321386,38.0
1,Aguadas,2006-12-31,2006,1100.508614,1113.070667,38.0
2,Aguadas,2007-12-31,2007,1037.679544,1100.944579,38.0
3,Aguadas,2008-12-31,2008,1080.767000,1084.587614,38.0
4,Aguadas,2009-12-31,2009,1125.788456,1129.578351,38.0
5,Aguadas,2010-12-31,2010,1030.396772,1132.272298,38.0
6,Aguadas,2011-12-31,2011,1111.471070,1112.413912,38.0
7,Aguadas,2012-12-31,2012,1156.204404,1168.532561,38.0
8,Aguadas,2013-12-31,2013,1092.135053,1145.374333,38.0
9,Aguadas,2014-12-31,2014,1135.022018,1158.682053,38.0


In [28]:
ruta_salida_mes = BASE_DIR / "data" / "processed" / "evapotranspiracion_caldas_mes_2005-2025.csv"
et_mes.to_csv(ruta_salida_mes, index=False)
ruta_salida_ano = BASE_DIR / "data" / "processed" / "evapotranspiracion_caldas_anio_2005-2025.csv"
et_anual.to_csv(ruta_salida_ano, index=False)